# MUSIC CLASSIFICATION THROUGH CNN

In [ ]:
import json
import os
import math
import librosa
import subprocess
import time
import warnings

DATASET_PATH = "/kaggle/input/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/Data/genres_original"
JSON_PATH = "/kaggle/working/data_10.json"

SAMPLE_RATE = 22050
TRACK_DURATION = 30 # measured in seconds
SAMPLES_PER_TRACK = SAMPLE_RATE * TRACK_DURATION

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

### Dataset Prepaation

In [ ]:
def save_mfcc(
    dataset_path,
    json_path,
    num_mfcc=13,
    n_fft=2048,
    hop_length=512,
    num_segments=10
):
    # extracting mfccs and saving them
    data = {
        "mapping": [],
        "labels": [],
        "mfcc": []
    }
    
    samples_per_segment = int(SAMPLES_PER_TRACK/num_segments)
    num_mfcc_vectors_per_segment = math.ceil(samples_per_segment/hop_length)
    
    corrupt_count = 0
    
    print("=" * 80)
    print("STARTING MFCC EXTRACTION")
    print("=" * 80)

    # Get genre folders explicitly
    genre_folders = sorted([
        folder for folder in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, folder))
    ])

    print(f"Dataset: {dataset_path}")
    print(f"Genres found: {genre_folders}")
    print()

    # Create mapping based on genre names
    label_mapping = {
        genre: i for i, genre in enumerate(genre_folders)
    }

    data["mapping"] = genre_folders

    # Process each genre
    for genre in genre_folders:

        dirpath = os.path.join(dataset_path, genre)
        label = label_mapping[genre]

        print(f"\nProcessing: {genre}")

        filenames = sorted([
            f for f in os.listdir(dirpath)
            if f.lower().endswith(".wav")
        ])

        process_count = 0

        for f in filenames:

            file_path = os.path.join(dirpath, f)

            # Load audio
            try:
                signal, sample_rate = librosa.load(
                    file_path,
                    sr=SAMPLE_RATE
                )

            except Exception:
                print(f"\nSkipping corrupted file: {file_path}")
                corrupt_count += 1
                continue

            # Divide song into segments
            for d in range(num_segments):

                start = samples_per_segment * d
                finish = start + samples_per_segment

                # Extract MFCC
                mfcc = librosa.feature.mfcc(
                    y=signal[start:finish],
                    sr=sample_rate,
                    n_mfcc=num_mfcc,
                    n_fft=n_fft,
                    hop_length=hop_length
                )

                mfcc = mfcc.T

                # Only keep segments with expected dimensions
                if len(mfcc) == num_mfcc_vectors_per_segment:

                    data["mfcc"].append(mfcc.tolist())
                    data["labels"].append(label)

                    process_count += 1

            if process_count % 10 == 0:
                print(
                    f"\r  {f} | {process_count} segments",
                    end=""
                )

        print(f"\n  Completed {genre}: {process_count} segments")

    # Save JSON
    with open(json_path, "w") as fp:
        json.dump(data, fp, indent=4)

    print("\n" + "=" * 80)
    print("MFCC EXTRACTION COMPLETE")
    print("=" * 80)

    print(f"JSON saved to: {json_path}")
    print(f"Total segments: {len(data['mfcc'])}")
    print(f"Total labels: {len(data['labels'])}")
    print(f"Genres: {data['mapping']}")

    if corrupt_count:
        print(f"Corrupted files skipped: {corrupt_count}")
    

In [ ]:
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset not found at:\n{DATASET_PATH}"
    )

save_mfcc(
    DATASET_PATH,
    JSON_PATH,
    num_segments=10
)

### CNN for Music Genre Classificatiion

In [ ]:
from sklearn.model_selection import train_test_split
import tensorflow.keras as keras
import numpy as np

DATA_PATH = "/kaggle/working/data_10.json"

def load_data(data_path):
    #Loads the MFCC dataset from a JSON file.

    with open(data_path, "r") as fp:
        data = json.load(fp)

    # Convert lists to NumPy arrays
    X = np.array(data["mfcc"])
    y = np.array(data["labels"])

    print("Data successfully loaded!")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")

    return X, y


# Load data
X, y = load_data(DATA_PATH)

In [ ]:
 # preparing dataset with validation set

def prepare_datasets(test_size, validation_size):

    #  loading data
    X, y = load_data(DATA_PATH)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size)

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=validation_size)


    # 3d array
    X_train = X_train[..., np.newaxis]
    X_val = X_val[..., np.newaxis]
    X_test = X_test[..., np.newaxis]


    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
def build_model(input_shape):


    # build network topology
    model = keras.Sequential()

    # 1st conv layer
    model.add(keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(keras.layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same'))
    model.add(keras.layers.BatchNormalization())

    # 2nd conv layer
    model.add(keras.layers.Conv2D(32, (3, 3), activation='relu'))
    model.add(keras.layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same'))
    model.add(keras.layers.BatchNormalization())

    # 3rd conv layer
    model.add(keras.layers.Conv2D(32, (2, 2), activation='relu'))
    model.add(keras.layers.MaxPooling2D((2, 2), strides=(2, 2), padding='same'))
    model.add(keras.layers.BatchNormalization())

    # flatten output and feed it into dense layer
    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dense(64, activation='relu'))
    model.add(keras.layers.Dropout(0.3))

    # output layer
    model.add(keras.layers.Dense(10, activation='softmax'))

    return model

In [ ]:
def predict(model, X, y):

    # add a dimension to input data for sample - model.predict() expects a 4d array in this case
    X = X[np.newaxis, ...] # array shape (1, 130, 13, 1)

    # perform prediction
    prediction = model.predict(X)

    # get index with max value
    predicted_index = np.argmax(prediction, axis=1)

    print(f"Target: {y}, Predicted label: {predicted_index}")

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history):
    fig, axs = plt.subplots(2,figsize=(10, 6))


    # create accuracy subplot
    axs[0].plot(history.history["accuracy"], label = "train accuracy")
    axs[0].plot(history.history["val_accuracy"], label = "test accuracy")
    axs[0].set_ylabel("Accuracy")
    axs[0].legend(loc="lower right")
    axs[0].set_title("Accuracy eval")

    # create error subplot
    axs[1].plot(history.history["loss"], label = "train error")
    axs[1].plot(history.history["val_loss"], label = "test error")
    axs[1].set_ylabel("Error")
    axs[1].set_xlabel("Epoch")
    axs[1].legend(loc="upper right")
    axs[1].set_title("Error eval")

    plt.show()



In [ ]:
# get train, validation, test splits
X_train, X_validation, X_test, y_train, y_validation, y_test = prepare_datasets(0.25, 0.2)

# create network
input_shape = (X_train.shape[1], X_train.shape[2], 1)
model = build_model(input_shape)

# compile model
optimiser = keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=optimiser,
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

model.summary()

# train model
history = model.fit(X_train, y_train, validation_data=(X_validation, y_validation), batch_size=32, epochs=30)

# plot accuracy/error for training and validation
plot_history(history)

# evaluate model on test set
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

# pick a sample to predict from the test set
X_to_predict = X_test[100]
y_to_predict = y_test[100]

# predict sample
predict(model, X_to_predict, y_to_predict)